### Phase 2 — Data Validation and Cleaning

Read raw datasets, clean them, and save the results separately.


In [ ]:
import pandas as pd
import json
from pathlib import Path
from pandas.api.types import is_numeric_dtype

rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


### <b>Phase 2 - Data Validation and Cleaning</b>
- <b>2.1 Profile:</b> Inspect row/column counts, inferred data types, sample records, missing values, and ranges.
- <b>2.2 Missing values:</b> Choose drop, fill, or retain, explaining why; do not indiscriminately delete rows.
- <b>2.3 Duplicates:</b> Find duplicates using stable identifiers (product URL / product ID / date), not only entire-row equality.
- <b>2.4 Text and numbers:</b> Convert book prices and rating strings into meaningful numeric fields; handle currency symbols and separators.
- <b>2.5 Schemas:</b> Use consistent snake_case headers and validate price, rating, date, and stock OHLCV types.
- <b>2.6 Export:</b> Write cleaned product and book CSVs; export at least one cleaned dataset to XML as a file-format exercise.
- <b>2.7 Quality log:</b> Record counts before/after cleaning, transformations, excluded rows, and unresolved concerns.
- Suggested cleaned outputs: data/clean/books_clean.csv, data/clean/products_clean.csv, data/clean/amzn_clean.csv, data/clean/products_clean.xml. Keep the original raw extracts unchanged.

In [ ]:
from pandas.api.types import is_numeric_dtype

amzn_raw = pd.read_csv("../data/raw/amzn_raw.csv")
products_raw = pd.read_csv("../data/raw/products_raw.csv")
book_raw = pd.read_csv("../data/raw/books_raw.csv")

# 2.1 Data profiling
# ----------------------------------------

# # Amazon Stocks:
# print(amzn_raw.shape)  # row/columns counts
# print(amzn_raw.dtypes)  # data types
# print(amzn_raw.head()) # sample records
# print(amzn_raw.isna().sum())   # missing values
# print(amzn_raw.describe(include="all")) # ranges


# # Products:
# print(products_raw.shape) # row/columns counts
# print(products_raw.dtypes) # data types
# print(products_raw.head()) # sample records
# print(products_raw.isna().sum()) # missing values
# print(products_raw.describe(include="all")) # ranges

# # Books:

# print(book_raw.shape) # row/columns counts
# print(book_raw.dtypes) # data types
# print(book_raw.head()) # sample records
# print(book_raw.isna().sum()) # missing values
# print(book_raw.describe(include="all")) # ranges



# 2.2 Missing Values
# ----------------------------

amzn_clean = amzn_raw.copy()
book_clean = book_raw.copy()
products_clean = products_raw.copy()

# Chech if there are missing values in the rows, I use boolean indexing

print(book_clean[book_clean.isna().any(axis=1)])

print(amzn_clean[amzn_clean.isna().any(axis=1)])

print(products_clean[products_clean.isna().any(axis=1)])

# We don't have to delete Missing values becuase they don't exist :)


# 2.2 Check duplicates
# ----------------------------

print(book_clean.duplicated(subset=["AbsoluteUrl"]).sum())
print(products_clean.duplicated(subset=["id"]).sum())
print(amzn_clean.duplicated(subset=["Date"]).sum())


# 2.3 Text and numbers in books:
# ----------------------------

book_clean["Price"] = pd.to_numeric(
        book_clean["Price"].str.replace("Â", "", regex=False).str.replace("£", "", regex=False),
        errors="coerce")

book_clean["Rating"] = book_clean["Rating"].map(rating_map)

# print(book_clean["Price"])




# 2.4 Use consistent snake_case headers and validate price, rating, date, and stock OHLCV types.
# ----------------------------






### Save cleaned data

These transformations and exports were moved out of Phase 1. Book price and rating transformations are already in the preceding Phase 2 cell.


In [ ]:
clean_path = Path("../data/clean")
clean_path.mkdir(parents=True, exist_ok=True)

# Books
book_clean.to_csv(clean_path / "books_clean.csv", index=False)

# Products
df = products_raw.copy()
required_product_cols = ["id", "title", "price", "category", "rating.rate", "rating.count"]
missing = [col for col in required_product_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required product fields: {missing}")

df = df[required_product_cols].rename(columns={
    "rating.rate": "rating_rate",
    "rating.count": "rating_count"
})

with open(clean_path / "products_clean.json", "w", encoding="utf-8") as file:
    json.dump(df.to_dict(orient="records"), file, indent=4, ensure_ascii=False)
df.to_csv(clean_path / "products_clean.csv", index=False)

# Amazon
df = amzn_raw.copy()
required_cols = ["Date", "Open", "High", "Low", "Close", "Volume"]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    print(f"Missing required AMZN columns: {missing}; clean file was not overwritten")
else:
    df[required_cols].to_csv(clean_path / "amzn_clean.csv", index=False)
